## Linear Regression
### Problem: Predict the resale price of a HDB based on its statistics

### Table of Contents
* [Importing Dependencies](#idp)
* [Importing Dataset](#ids)
* [Analysing Data](#ad)
* [Optimizing Data](#od)
* [Training and testing the model](#ttm)
* [Using the model](#utm)

### Importing Dependencies <a id = "idp">

In [1]:
# Importing Packages
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import category_encoders as ce

### Importing Dataset <a id = "ids">

In [5]:
# Importing dataset
df_1 = pd.read_csv('resale_data_1.csv')
df_2 = pd.read_csv('resale_data_2.csv')
df_3 = pd.read_csv('resale_data_3.csv')
df = pd.concat([df_1, df_2, df_3])
df

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
0,2012-03,ANG MO KIO,2 ROOM,172,ANG MO KIO AVE 4,06 TO 10,45.0,Improved,1986,250000.0
1,2012-03,ANG MO KIO,2 ROOM,510,ANG MO KIO AVE 8,01 TO 05,44.0,Improved,1980,265000.0
2,2012-03,ANG MO KIO,3 ROOM,610,ANG MO KIO AVE 4,06 TO 10,68.0,New Generation,1980,315000.0
3,2012-03,ANG MO KIO,3 ROOM,474,ANG MO KIO AVE 10,01 TO 05,67.0,New Generation,1984,320000.0
4,2012-03,ANG MO KIO,3 ROOM,604,ANG MO KIO AVE 5,06 TO 10,67.0,New Generation,1980,321000.0
...,...,...,...,...,...,...,...,...,...,...
275522,2012-02,YISHUN,5 ROOM,212,YISHUN ST 21,10 TO 12,121.0,Improved,1985,476888.0
275523,2012-02,YISHUN,5 ROOM,758,YISHUN ST 72,01 TO 03,122.0,Improved,1986,490000.0
275524,2012-02,YISHUN,5 ROOM,873,YISHUN ST 81,01 TO 03,122.0,Improved,1988,488000.0
275525,2012-02,YISHUN,EXECUTIVE,664,YISHUN AVE 4,07 TO 09,181.0,Apartment,1992,705000.0


### Analyzing Data <a id = "ad">

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 826581 entries, 0 to 275526
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                826581 non-null  object 
 1   town                 826581 non-null  object 
 2   flat_type            826581 non-null  object 
 3   block                826581 non-null  object 
 4   street_name          826581 non-null  object 
 5   storey_range         826581 non-null  object 
 6   floor_area_sqm       826581 non-null  float64
 7   flat_model           826581 non-null  object 
 8   lease_commence_date  826581 non-null  int64  
 9   resale_price         826581 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 69.4+ MB


In [7]:
df.describe()

,floor_area_sqm,lease_commence_date,resale_price
count,826581.000000,826581.000000,8.265810e+05
mean,95.557909,1987.149345,2.935490e+05
std,26.057093,9.451743,1.487536e+05
min,28.000000,1966.000000,5.000000e+03
25%,73.000000,1980.000000,1.810000e+05
50%,93.000000,1986.000000,2.750000e+05
75%,114.000000,1994.000000,3.830000e+05
max,307.000000,2019.000000,1.258000e+06


### Optimizing Data <a id = "od">

In [8]:
#Dropping columns 'block' and 'street_name' as they have too much cardinality to be processed
df2 = df.drop(['block','street_name'], axis = 1)
df2

,month,town,flat_type,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
0,2012-03,ANG MO KIO,2 ROOM,06 TO 10,45.0,Improved,1986,250000.0
1,2012-03,ANG MO KIO,2 ROOM,01 TO 05,44.0,Improved,1980,265000.0
2,2012-03,ANG MO KIO,3 ROOM,06 TO 10,68.0,New Generation,1980,315000.0
3,2012-03,ANG MO KIO,3 ROOM,01 TO 05,67.0,New Generation,1984,320000.0
4,2012-03,ANG MO KIO,3 ROOM,06 TO 10,67.0,New Generation,1980,321000.0
...,...,...,...,...,...,...,...,...
275522,2012-02,YISHUN,5 ROOM,10 TO 12,121.0,Improved,1985,476888.0
275523,2012-02,YISHUN,5 ROOM,01 TO 03,122.0,Improved,1986,490000.0
275524,2012-02,YISHUN,5 ROOM,01 TO 03,122.0,Improved,1988,488000.0
275525,2012-02,YISHUN,EXECUTIVE,07 TO 09,181.0,Apartment,1992,705000.0


#### Processing the storey_range column

In [9]:
df2['storey_range'].value_counts()

04 TO 06    209705
07 TO 09    189109
01 TO 03    168985
10 TO 12    160250
13 TO 15     52212
16 TO 18     19449
19 TO 21      9422
22 TO 24      6053
01 TO 05      2700
25 TO 27      2544
06 TO 10      2474
11 TO 15      1259
28 TO 30      1049
34 TO 36       267
16 TO 20       265
31 TO 33       265
37 TO 39       255
40 TO 42       132
21 TO 25        92
26 TO 30        39
46 TO 48        21
43 TO 45        16
49 TO 51         9
36 TO 40         7
31 TO 35         2
Name: storey_range, dtype: int64

In [10]:
df2

,month,town,flat_type,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price
0,2012-03,ANG MO KIO,2 ROOM,06 TO 10,45.0,Improved,1986,250000.0
1,2012-03,ANG MO KIO,2 ROOM,01 TO 05,44.0,Improved,1980,265000.0
2,2012-03,ANG MO KIO,3 ROOM,06 TO 10,68.0,New Generation,1980,315000.0
3,2012-03,ANG MO KIO,3 ROOM,01 TO 05,67.0,New Generation,1984,320000.0
4,2012-03,ANG MO KIO,3 ROOM,06 TO 10,67.0,New Generation,1980,321000.0
...,...,...,...,...,...,...,...,...
275522,2012-02,YISHUN,5 ROOM,10 TO 12,121.0,Improved,1985,476888.0
275523,2012-02,YISHUN,5 ROOM,01 TO 03,122.0,Improved,1986,490000.0
275524,2012-02,YISHUN,5 ROOM,01 TO 03,122.0,Improved,1988,488000.0
275525,2012-02,YISHUN,EXECUTIVE,07 TO 09,181.0,Apartment,1992,705000.0


In [22]:
# Declaring the storey value as the estimated average of the storey range since the ranges vary
df2['low_s'] = df2['storey_range'].str[:2]
df2['high_s'] = df2['storey_range'].str[-2:]
df2['low_s'] = df2['low_s'].astype(int)
df2['high_s'] = df2['high_s'].astype(int)
df2a = df2.copy()
df2a['storey'] = df2a[['high_s', 'low_s']].mean(axis = 1)
df3 = df2a.drop(['storey_range', 'low_s', 'high_s'], axis = 1)
df3

,month,town,flat_type,floor_area_sqm,flat_model,lease_commence_date,resale_price,storey
0,2012-03,ANG MO KIO,2 ROOM,45.0,Improved,1986,250000.0,8.0
1,2012-03,ANG MO KIO,2 ROOM,44.0,Improved,1980,265000.0,3.0
2,2012-03,ANG MO KIO,3 ROOM,68.0,New Generation,1980,315000.0,8.0
3,2012-03,ANG MO KIO,3 ROOM,67.0,New Generation,1984,320000.0,3.0
4,2012-03,ANG MO KIO,3 ROOM,67.0,New Generation,1980,321000.0,8.0
...,...,...,...,...,...,...,...,...
275522,2012-02,YISHUN,5 ROOM,121.0,Improved,1985,476888.0,11.0
275523,2012-02,YISHUN,5 ROOM,122.0,Improved,1986,490000.0,2.0
275524,2012-02,YISHUN,5 ROOM,122.0,Improved,1988,488000.0,2.0
275525,2012-02,YISHUN,EXECUTIVE,181.0,Apartment,1992,705000.0,8.0


#### Processing the year column

In [23]:
# Extracting just the year from the "month" column
df3['year'] = df3['month'].str[:4]
df3['year'] = df3['year'].astype(int)
df4 = df3.drop(['month'], axis = 1)
df4

,town,flat_type,floor_area_sqm,flat_model,lease_commence_date,resale_price,storey,year
0,ANG MO KIO,2 ROOM,45.0,Improved,1986,250000.0,8.0,2012
1,ANG MO KIO,2 ROOM,44.0,Improved,1980,265000.0,3.0,2012
2,ANG MO KIO,3 ROOM,68.0,New Generation,1980,315000.0,8.0,2012
3,ANG MO KIO,3 ROOM,67.0,New Generation,1984,320000.0,3.0,2012
4,ANG MO KIO,3 ROOM,67.0,New Generation,1980,321000.0,8.0,2012
...,...,...,...,...,...,...,...,...
275522,YISHUN,5 ROOM,121.0,Improved,1985,476888.0,11.0,2012
275523,YISHUN,5 ROOM,122.0,Improved,1986,490000.0,2.0,2012
275524,YISHUN,5 ROOM,122.0,Improved,1988,488000.0,2.0,2012
275525,YISHUN,EXECUTIVE,181.0,Apartment,1992,705000.0,8.0,2012


#### Processing flat model column

In [24]:
df4['flat_model'].value_counts()

Model A                   158008
Improved                  143763
New Generation             98672
NEW GENERATION             78898
IMPROVED                   73593
MODEL A                    70381
Premium Apartment          35029
Simplified                 30702
SIMPLIFIED                 23258
Standard                   22479
Apartment                  22103
STANDARD                   17375
Maisonette                 14694
MAISONETTE                 12215
APARTMENT                   9901
Model A2                    9109
DBSS                        1609
Adjoined flat               1085
MODEL A-MAISONETTE           982
Model A-Maisonette           907
Terrace                      395
MULTI GENERATION             279
Type S1                      272
TERRACE                      247
Multi Generation             223
Type S2                      129
Premium Maisonette            82
Improved-Maisonette           70
IMPROVED-MAISONETTE           44
Premium Apartment Loft        31
2-ROOM    

In [25]:
# Compiling the similar values flat model into one value
df5 = df4
df5['flat_model'] = df5['flat_model'].str.lower().str.strip()
df5['flat_model'].value_counts()

model a                   228389
improved                  217356
new generation            177570
simplified                 53960
standard                   39854
premium apartment          35035
apartment                  32004
maisonette                 26909
model a2                    9109
model a-maisonette          1889
dbss                        1609
adjoined flat               1085
terrace                      642
multi generation             502
type s1                      272
type s2                      129
improved-maisonette          114
premium maisonette            82
2-room                        40
premium apartment loft        31
Name: flat_model, dtype: int64

#### Processing flat type column

In [26]:
df5['flat_type'].value_counts()

4 ROOM              309314
3 ROOM              272580
5 ROOM              170408
EXECUTIVE            62641
2 ROOM                9863
1 ROOM                1273
MULTI GENERATION       279
MULTI-GENERATION       223
Name: flat_type, dtype: int64

In [27]:
# Compiling the similar flat types into one value and mapping them to numerical values
df6 = df5
df6.loc[df6['flat_type'] == 'MULTI-GENERATION', 'flat_type'] = 'MULTI GENERATION'

type_mapping = {
    '1 ROOM': 0,
    '2 ROOM': 1,
    '3 ROOM': 2,
    '4 ROOM': 3,
    '5 ROOM': 4,
    'EXECUTIVE':5,
    'MULTI GENERATION':6
}
df6['flat_type_enc'] = df6['flat_type'].map(type_mapping)
df7 = df6.drop(['flat_type'], axis = 1)
df7

,town,floor_area_sqm,flat_model,lease_commence_date,resale_price,storey,year,flat_type_enc
0,ANG MO KIO,45.0,improved,1986,250000.0,8.0,2012,1
1,ANG MO KIO,44.0,improved,1980,265000.0,3.0,2012,1
2,ANG MO KIO,68.0,new generation,1980,315000.0,8.0,2012,2
3,ANG MO KIO,67.0,new generation,1984,320000.0,3.0,2012,2
4,ANG MO KIO,67.0,new generation,1980,321000.0,8.0,2012,2
...,...,...,...,...,...,...,...,...
275522,YISHUN,121.0,improved,1985,476888.0,11.0,2012,4
275523,YISHUN,122.0,improved,1986,490000.0,2.0,2012,4
275524,YISHUN,122.0,improved,1988,488000.0,2.0,2012,4
275525,YISHUN,181.0,apartment,1992,705000.0,8.0,2012,5


#### Splitting the data into train test sets before encoding to prevent data leakage

In [28]:
X = df7.drop('resale_price', axis=1)
y = df7['resale_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Encoding and Scaling the data

In [29]:
# Target Encoding 'town' and 'flat_model' to link those values to the resale price
encoder = ce.TargetEncoder(cols=['town'])
X_train['town_encoded'] = encoder.fit_transform(X_train['town'], y_train)
X_test['town_encoded'] = encoder.transform(X_test['town'])

enc2 = ce.TargetEncoder(cols = ['flat_model'])
X_train['flat_model_enc']  =enc2.fit_transform(X_train['flat_model'],y_train)
X_test['flat_model_enc'] = enc2.transform(X_test['flat_model'])

#Dropping the original columns
X_train = X_train.drop(['town', 'flat_model'], axis = 1)
X_test = X_test.drop(['town', 'flat_model'], axis = 1)

In [30]:
# Scaling the dataset to prevent overpowering features
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

### Training and testing the model <a id = "ttm">

In [31]:
# Creating and testing the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

# Testing Metrics
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Train R2 Score: {r2_train:.4f}")
print(f"MAE: ${mae_train:,.2f}")

print(f"Test R2 Score: {r2_test:.4f}")
print(f"Test MAE: ${mae_test:,.2f}")

Train R2 Score: 0.7355
MAE: $56,591.05
Test R2 Score: 0.7346
Test MAE: $56,643.74


### Using the model <a id = "utm">

In [32]:
hdb_info = {
    'town' :"ANG MO KIO",
    'floor_area_sqm': 73,
    'flat_model': "improved",
    'lease_commence_date' : 1986,
    'storey': 8,
    'year':2012,
    'flat_type_enc':1
}
hdb_df = pd.DataFrame([hdb_info])

In [33]:
hdb_df['town_encoded'] = encoder.transform(hdb_df['town'])
hdb_df['flat_model_enc'] = enc2.transform(hdb_df['flat_model'])
hdb_df2 = hdb_df.drop(['town', 'flat_model'], axis = 1)
hdb_df3 = pd.DataFrame(scaler.transform(hdb_df2), columns=hdb_df2.columns, index=hdb_df2.index)

predicted_price = model.predict(hdb_df3)
print(f"Predicted Resale Price: ${predicted_price[0]:,.2f}")

Predicted Resale Price: $258,922.21
